## 2 · Initial data exploration

### By:
Sofía Arango Patiño

### Date:
2026-11-01

### Description:

Second step of the proof of concept ([issue #2](https://github.com/modsiw-mona/anomaly_detection_ato/issues/2);
CRISP-DM phase 2, *data understanding*; thesis objective OE1).

The raw CSV from issue #1 stores everything as text. This notebook:

1. **profiles the raw values** column by column, to find how missing values are written;
2. **decides** the null markers and the storage type of every field, *from what was measured* — and records
   the decisions in the dataset's translation file (`conf/dataset/rba.yaml`), not in this notebook;
3. **builds the intermediate dataset** (`data/02_intermediate/rba.parquet`): standard names, unified nulls,
   real types, artefact flags;
4. **proves nothing was lost**, by recomputing every count on both files;
5. gives a first overview of the typed data.

The conversion code lives in `src/data/intermediate.py` (tested in `tests/data/test_intermediate.py`), so the
feature pipeline can reuse it later.

## 📚 Import libraries

In [1]:
import sys
from pathlib import Path

# Make the project's `src/` importable, wherever the notebook is opened from.
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

In [2]:
import logging
import resource
import time

import polars as pl

from data.datasets import load_dataset_config
from data.intermediate import build_intermediate, intermediate_path, scan_raw
from data.schema import FIELDS_BY_NAME

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)
pl.Config.set_tbl_rows(30)
pl.Config.set_fmt_str_lengths(50)
pl.Config.set_tbl_width_chars(140)

polars.config.Config

## 💾 Load data

In [3]:
config = load_dataset_config("rba")
data_dir = REPO_ROOT / "data"
raw_file = data_dir / "01_raw" / Path(config.source.member).name
raw = scan_raw(raw_file)
rename = config.rename_map()  # column in file -> standard name
print(f"raw file: {raw_file.relative_to(REPO_ROOT)}  ({raw_file.stat().st_size / 1e9:.2f} GB)")

raw file: data/01_raw/rba-dataset.csv  (9.05 GB)


## 🔎 How are missing values written?

Every mapped column is scanned once (streaming) for true nulls, empty strings, and the usual textual stand-ins
for "missing". A non-zero count is a *candidate*, not a verdict: each one is checked before it becomes a rule.

In [4]:
CANDIDATES = ["-", "?", "null", "none", "nan", "unknown", "n/a", "na"]

exprs = []
for column in rename:
    text = pl.col(column).str.strip_chars().str.to_lowercase()
    exprs += [
        pl.col(column).null_count().alias(f"{column}#null"),
        (pl.col(column) == "").sum().alias(f"{column}#empty"),
        *((text == m).sum().alias(f"{column}={m}") for m in CANDIDATES),
    ]
start = time.perf_counter()
scan = raw.select(exprs).collect(engine="streaming").row(0, named=True)
print(f"scanned in {time.perf_counter() - start:.0f} s")

profile = pl.DataFrame(
    [
        {
            "column in file": column,
            "standard name": rename[column],
            "null": scan[f"{column}#null"],
            "empty": scan[f"{column}#empty"],
            "candidate markers": ", ".join(
                f"'{m}': {scan[f'{column}={m}']:,}" for m in CANDIDATES if scan[f"{column}={m}"]
            )
            or "—",
        }
        for column in rename
    ]
)
profile

scanned in 23 s


column in file,standard name,null,empty,candidate markers
str,str,i64,i64,str
"""User ID""","""user_id""",0,0,"""—"""
"""Login Timestamp""","""timestamp""",0,0,"""—"""
"""Login Successful""","""login_success""",0,0,"""—"""
"""IP Address""","""ip""",0,0,"""—"""
"""ASN""","""asn""",0,0,"""—"""
"""Device Type""","""device_type""",1526,0,"""'unknown': 867,371"""
"""User Agent String""","""user_agent""",0,0,"""—"""
"""Country""","""country""",0,0,"""—"""
"""Region""","""region""",47409,0,"""'-': 14,013,381, 'nan': 3"""


Three candidates appear. Each is checked against the data and the dataset's documentation:

In [5]:
region, device = config.columns["region"], config.columns["device_type"]
print("Rows whose region reads 'nan' (any case):")
print(
    raw.filter(pl.col(region).str.to_lowercase() == "nan")
    .select(config.columns["country"], region, config.columns["city"])
    .collect(engine="streaming")
)
print("\nDevice types:")
print(raw.select(pl.col(device).value_counts(sort=True)).collect(engine="streaming").unnest(device))

Rows whose region reads 'nan' (any case):


shape: (3, 3)
┌─────────┬────────┬──────┐
│ Country ┆ Region ┆ City │
│ ---     ┆ ---    ┆ ---  │
│ str     ┆ str    ┆ str  │
╞═════════╪════════╪══════╡
│ TH      ┆ Nan    ┆ Pua  │
│ TH      ┆ Nan    ┆ Pua  │
│ TH      ┆ Nan    ┆ Pua  │
└─────────┴────────┴──────┘

Device types:


shape: (6, 2)
┌─────────────┬──────────┐
│ Device Type ┆ count    │
│ ---         ┆ ---      │
│ str         ┆ u32      │
╞═════════════╪══════════╡
│ mobile      ┆ 19642650 │
│ desktop     ┆ 7934515  │
│ bot         ┆ 2027834  │
│ unknown     ┆ 867371   │
│ tablet      ┆ 795368   │
│ null        ┆ 1526     │
└─────────────┴──────────┘


**Decisions** — recorded under `null_markers` in `conf/dataset/rba.yaml`:

| Candidate | Where | Verdict | Why |
| --- | --- | --- | --- |
| `-` | Region, City | **null marker** | A placeholder for "no region / no city", on ~45% of rows. |
| `nan` | Region (3 rows) | **real value — kept** | All three rows are *Nan* province, Thailand (country `TH`, city `Pua`). |
| `unknown` | Device Type | **real category — kept** | Listed by the dataset authors as a device type (`mobile`, `desktop`, `tablet`, `bot`, `unknown`). |
| true nulls | Device Type (1,526) | **kept as null** | The authors' README: user agents that could not be parsed; "perhaps this parse error is useful". |

This is why null markers are declared **per column**: a global "`nan` means missing" rule would have deleted a
real Thai province, and a global "`unknown` means missing" rule would have deleted a real device category.

## 🧮 What type is each field?

Measured formats, all in one streaming pass:

In [6]:
c = config.columns
formats = (
    raw.select(
        pl.col(c["timestamp"]).str.len_chars().n_unique().alias("timestamp: distinct lengths"),
        pl.col(c["timestamp"])
        .str.contains(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d{3}$")
        .all()
        .alias("timestamp: all 'YYYY-MM-DD hh:mm:ss.fff'"),
        pl.col(c["user_id"]).str.contains(r"^-?\d+$").all().alias("user_id: all integers"),
        pl.col(c["asn"]).str.contains(r"^\d+$").all().alias("asn: all digits"),
        pl.col(c["ip"]).str.contains(":").any().alias("ip: any IPv6"),
        pl.col(c["rtt_ms"])
        .drop_nulls()
        .str.contains(r"^\d+(\.\d+)?$")
        .all()
        .alias("rtt: all numeric"),
        pl.col(c["login_success"]).unique().sort().implode().alias("login_success: values"),
    )
    .collect(engine="streaming")
    .row(0, named=True)
)
pl.DataFrame({"check": list(formats), "result": [str(v) for v in formats.values()]})

check,result
str,str
"""timestamp: distinct lengths""","""1"""
"""timestamp: all 'YYYY-MM-DD hh:mm:ss.fff'""","""True"""
"""user_id: all integers""","""True"""
"""asn: all digits""","""True"""
"""ip: any IPv6""","""False"""
"""rtt: all numeric""","""True"""
"""login_success: values""","""['False', 'True']"""


**Decisions** — each *kind* of field gets one storage type (`src/data/intermediate.py`); per-dataset choices
live in `conf/dataset/rba.yaml`:

| Kind | Fields | Stored as | Why |
| --- | --- | --- | --- |
| identifier | `user_id` | `Int64` (`identifier_dtypes`) | All 64-bit integer pseudonyms here; other datasets default to text. |
| timestamp | `timestamp` | `Datetime[ms]` (`timestamp_format`) | A single format, millisecond precision. |
| boolean | `login_success`, labels | `Boolean` | Only `True` / `False`. |
| categorical | `ip`, `asn`, `country`, `region`, `city`, `os`, `browser`, `device_type` | `Categorical` | Values are *identities*. `asn` is digits, but ASN 3000 is not "bigger" than ASN 2000 — storing it as a number invites arithmetic that means nothing. |
| text | `user_agent` | `String` | Free text, ~370 k distinct values. |
| numeric | `rtt_ms` | `Float64` | Milliseconds, written like `1123.0`. |

Conversions are **strict**: a value that does not parse stops the build instead of silently becoming null. The
unit tests check this for every kind.

## 👷 Build the intermediate dataset

In [7]:
out_file = intermediate_path(config, data_dir)
start = time.perf_counter()
build_intermediate(config, raw_file, out_file)
elapsed = time.perf_counter() - start
peak_gb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e9  # bytes on macOS
print(f"built {out_file.relative_to(REPO_ROOT)} in {elapsed:.0f} s")
print(
    f"size: {out_file.stat().st_size / 1e6:,.0f} MB (raw CSV: {raw_file.stat().st_size / 1e6:,.0f} MB)"
)
print(f"peak memory of this notebook so far: {peak_gb:.1f} GB")

INFO Wrote rba.parquet


built data/02_intermediate/rba.parquet in 22 s
size: 676 MB (raw CSV: 9,053 MB)
peak memory of this notebook so far: 7.6 GB


## ✅ Did anything get lost?

Every count is computed **independently on both files** — the raw text and the typed parquet — and compared.
For the null-marked columns, the expected nulls are *true nulls + markers*.

In [8]:
typed = pl.scan_parquet(out_file)
col = config.columns
lab = config.labels


def is_true(name: str) -> pl.Expr:
    return pl.col(name).str.to_lowercase() == "true"


markers = {field: list(values) for field, values in config.null_markers.items()}
asn_threshold = next(a.value for a in config.artefacts if a.name == "asn_is_placeholder")
raw_counts = raw.select(
    pl.len().alias("rows"),
    pl.col(col["user_id"]).n_unique().alias("distinct users"),
    is_true(col["login_success"]).sum().alias("successful logins"),
    is_true(lab["label_ato"]).sum().alias("takeover labels"),
    is_true(lab["label_attack_ip"]).sum().alias("attack-IP labels"),
    (pl.col(col["region"]).is_null() | pl.col(col["region"]).is_in(markers["region"]))
    .sum()
    .alias("region missing"),
    (pl.col(col["city"]).is_null() | pl.col(col["city"]).is_in(markers["city"]))
    .sum()
    .alias("city missing"),
    pl.col(col["rtt_ms"]).null_count().alias("rtt missing"),
    pl.col(col["device_type"]).null_count().alias("device_type missing"),
    (pl.col(col["region"]) == "Nan").sum().alias("region 'Nan' rows"),
    (pl.col(col["asn"]).cast(pl.Int64) >= asn_threshold).sum().alias("ASN placeholder"),
    # Independent cross-check: a text prefix, unlike the integer range used by the module.
    pl.col(col["ip"]).str.starts_with("10.").sum().alias("IP placeholder"),
).collect(engine="streaming")

typed_counts = typed.select(
    pl.len().alias("rows"),
    pl.col("user_id").n_unique().alias("distinct users"),
    pl.col("login_success").sum().alias("successful logins"),
    pl.col("label_ato").sum().alias("takeover labels"),
    pl.col("label_attack_ip").sum().alias("attack-IP labels"),
    pl.col("region").null_count().alias("region missing"),
    pl.col("city").null_count().alias("city missing"),
    pl.col("rtt_ms").null_count().alias("rtt missing"),
    pl.col("device_type").null_count().alias("device_type missing"),
    (pl.col("region") == "Nan").sum().alias("region 'Nan' rows"),
    pl.col("asn_is_placeholder").sum().alias("ASN placeholder"),
    pl.col("ip_is_placeholder").sum().alias("IP placeholder"),
).collect()

comparison = pl.DataFrame(
    {
        "check": raw_counts.columns,
        "raw CSV": [int(v) for v in raw_counts.row(0)],
        "intermediate parquet": [int(v) for v in typed_counts.row(0)],
    }
).with_columns((pl.col("raw CSV") == pl.col("intermediate parquet")).alias("match"))
assert comparison["match"].all(), "the intermediate dataset differs from the raw file"
comparison

check,raw CSV,intermediate parquet,match
str,i64,i64,bool
"""rows""",31269264,31269264,true
"""distinct users""",4304857,4304857,true
"""successful logins""",12541442,12541442,true
"""takeover labels""",141,141,true
"""attack-IP labels""",3096977,3096977,true
"""region missing""",14060790,14060790,true
"""city missing""",13913142,13913142,true
"""rtt missing""",29993329,29993329,true
"""device_type missing""",1526,1526,true


In [9]:
order = typed.select(
    pl.col("timestamp").is_sorted().alias("timestamp in order"),
    pl.col("event_id").is_sorted().alias("event_id in order"),
).collect()
order

timestamp in order,event_id in order
bool,bool
true,true


## 📊 The intermediate dataset

### Schema, missingness and cardinality

In [10]:
overview = (
    typed.select(
        *(
            pl.col(name).null_count().alias(f"{name}|null")
            for name in typed.collect_schema().names()
        ),
        *(
            pl.col(name).n_unique().alias(f"{name}|unique")
            for name in typed.collect_schema().names()
        ),
    )
    .collect()
    .row(0, named=True)
)
rows = typed_counts["rows"].item()
schema = typed.collect_schema()
pl.DataFrame(
    {
        "field": list(schema.names()),
        "role": [
            FIELDS_BY_NAME[n].role.value
            if n in FIELDS_BY_NAME
            else ("id" if n == "event_id" else "artefact flag")
            for n in schema.names()
        ],
        "dtype": [str(t) for t in schema.dtypes()],
        "missing %": [round(100 * overview[f"{n}|null"] / rows, 2) for n in schema.names()],
        "distinct values": [overview[f"{n}|unique"] for n in schema.names()],
    }
)

field,role,dtype,missing %,distinct values
str,str,str,f64,i64
"""event_id""","""id""","""UInt32""",0.0,31269264
"""user_id""","""required""","""Int64""",0.0,4304857
"""timestamp""","""required""","""Datetime(time_unit='ms', time_zone=None)""",0.0,31269085
"""login_success""","""required""","""Boolean""",0.0,2
"""ip""","""context""","""Categorical""",0.0,3512330
"""asn""","""context""","""Categorical""",0.0,12955
"""device_type""","""context""","""Categorical""",0.0,6
"""user_agent""","""context""","""String""",0.0,420952
"""country""","""optional""","""Categorical""",0.0,229


### A few typed rows

In [11]:
typed.head(5).collect()

event_id,user_id,timestamp,login_success,ip,asn,device_type,user_agent,country,region,city,os,browser,rtt_ms,label_ato,label_attack_ip,asn_is_placeholder,ip_is_placeholder
u32,i64,datetime[ms],bool,cat,cat,cat,str,cat,cat,cat,cat,cat,f64,bool,bool,bool,bool
0,-4324475583306591935,2020-02-03 12:43:30.772,false,"""10.0.65.171""","""29695""","""mobile""","""Mozilla/5.0 (iPhone; CPU iPhone OS 13_4 like Mac …","""NO""",null,null,"""iOS 13.4""","""Firefox 20.0.0.1618""",null,false,false,false,true
1,-4324475583306591935,2020-02-03 12:43:43.549,false,"""194.87.207.6""","""60117""","""mobile""","""Mozilla/5.0 (Linux; Android 4.1; Galaxy Nexus Bui…","""AU""",null,null,"""Android 4.1""","""Chrome Mobile 46.0.2490""",null,false,false,false,false
2,-3284137479262433373,2020-02-03 12:43:55.873,true,"""81.167.144.58""","""29695""","""mobile""","""Mozilla/5.0 (iPad; CPU OS 7_1 like Mac OS X) Appl…","""NO""","""Vestland""","""Urangsvag""","""iOS 7.1""","""Android 2.3.3.2672""",null,false,false,false,false
3,-4324475583306591935,2020-02-03 12:43:56.180,false,"""170.39.78.152""","""393398""","""mobile""","""Mozilla/5.0 (Linux; Android 4.1; Galaxy Nexus Bui…","""US""",null,null,"""Android 4.1""","""Chrome Mobile WebView 85.0.4183""",null,false,false,false,false
4,-4618854071942621186,2020-02-03 12:43:59.396,false,"""10.0.0.47""","""398986""","""mobile""","""Mozilla/5.0 (Linux; U; Android 2.2) Build/NMA26.4…","""US""","""Virginia""","""Ashburn""","""Android 2.2""","""Chrome Mobile WebView 85.0.4183""",null,false,true,false,true


### Signals with partial coverage

In [12]:
coverage = (
    typed.group_by("login_success")
    .agg(
        pl.len().alias("attempts"),
        pl.col("rtt_ms").is_not_null().mean().mul(100).round(1).alias("rtt present %"),
        pl.col("region").is_not_null().mean().mul(100).round(1).alias("region present %"),
        pl.col("city").is_not_null().mean().mul(100).round(1).alias("city present %"),
        pl.col("ip_is_placeholder").mean().mul(100).round(1).alias("placeholder IP %"),
        pl.col("asn_is_placeholder").mean().mul(100).round(1).alias("placeholder ASN %"),
    )
    .sort("login_success")
    .collect()
)
coverage

login_success,attempts,rtt present %,region present %,city present %,placeholder IP %,placeholder ASN %
bool,u32,f64,f64,f64,f64,f64
false,18727822,2.1,50.1,50.5,28.7,5.1
true,12541442,7.0,62.4,63.0,15.3,4.1


## 📊 Analysis of Results and Conclusions

*(Numbers refer to the outputs above.)*

- **The intermediate dataset is complete and faithful.** Twelve independent counts — rows, users, successes, both
  labels, every kind of missing value, the artefact rules, and the three *Nan* rows — are identical on the raw CSV
  and on the parquet, and time order is preserved. The 141 takeover labels are intact.
- **Missing values are now real nulls, decided from evidence.** Only `-` in `region` and `city` was a null marker.
  `nan` turned out to be a real place and `unknown` a real device category; declaring markers *per column* is what
  kept them.
- **Types follow meaning, not appearance.** `asn` and `ip` are stored as categories (identities), not numbers;
  user IDs as 64-bit integers; time at millisecond precision. Every conversion is strict.
- **Size and cost on a 16 GB laptop:** the 9 GB CSV becomes a ~0.7 GB parquet in well under a minute, with peak
  memory far below the machine's limit. Later notebooks read the parquet, not the CSV.
- **Weak or partial signals — to be tested, not discarded** (development plan, principle 1):
  - **RTT is missing for ~96% of attempts** (present for ~7% of successful logins). Any RTT feature will be
    sparse; the RTT ablation in issue #6 starts with that prior.
  - **Region and city are missing for ~45% of attempts**, and the dataset authors state they do not keep the real
    relations; also an ablation candidate.
  - **23% of attempts carry a placeholder IP and ~5% a placeholder ASN — and not evenly:** placeholder IPs are
    almost **twice as frequent in failed logins (28.7%) as in successful ones (15.3%)**. Since failed logins are
    dominated by attacks, the synthesis artefact itself is correlated with attack activity. A model could learn
    "placeholder IP ⇒ suspicious" — a property of how the data was *generated*, not of attacker *behaviour*.
    Issue #3 must measure the link between the placeholder flags and both labels, and issue #6 runs the planned
    ablation with and without placeholder rows. Behaviour features (new-for-this-user, rarity) do not use the
    placeholder *values*, but a placeholder shared by many users would look like a "shared IP", which #3 also
    checks.

## 💡 Proposals and Ideas

- **Issue #3 (EDA):** work from `data/02_intermediate/rba.parquet`. Priorities: the 140 takeovers (when, which
  users, failed attempts just before, attack-IP overlap); how strongly the placeholder flags relate to login success, attack IPs and takeovers; how many users share each placeholder IP/ASN; whether
  missing RTT/region/city is related to the labels (missingness itself can be a signal); the time distribution of
  takeovers, which decides the final test split by the rule in the development plan; and the validation rules
  (Pandera) for the feature pipeline.
- Turn the checks of this notebook into automated data-validation rules, so the same guarantees hold every time the
  pipeline runs (course part 2, task 2).

## 📖 References

- Wiefling, S., Jørgensen, P. R., Thunem, S., & Lo Iacono, L. (2022). Pump Up Password Security! *ACM TOPS*, 26(1).
  https://doi.org/10.1145/3546069 — and the dataset README, "Data Creation" and footnote on device types.
- Kedro data layer convention (raw → intermediate): https://docs.kedro.org/en/stable/faq/faq.html
- Polars user guide, streaming and categorical data: https://docs.pola.rs/
- Course methodology: https://joserzapata.github.io/post/ciencia-datos-proyecto-python/2-exploration/